In [1]:
import re
import numpy as np
import os
import torch
import tempfile

from pathlib import Path
from datasets import load_dataset
from moviepy.video.io.VideoFileClip import VideoFileClip
from IPython.display import Video, display
from safetensors.torch import load_file

from src.config import Config
from src.model.virality_predictor import ViralityPredictor
from src.model.data_processor import DataProcessor

if torch.backends.mps.is_available():
    device = torch.device('mps')
    print('Using MPS.')
else:
    device = torch.device('cpu')
    print('Using CPU.')

config = Config()
checkpoint_dir = Path(config.checkpoint_path)
checkpoint_folders = [f for f in checkpoint_dir.iterdir(
) if f.is_dir() and re.match(r'checkpoint-\d+$', f.name)]
latest_checkpoint = max(checkpoint_folders, key=lambda f: int(
    f.name.split('-')[1])) if checkpoint_folders else None
assert latest_checkpoint, f'No checkpoint found inside {checkpoint_dir}'

print(f'Loading model from: {latest_checkpoint}.')
model = ViralityPredictor(config)
state_dict = load_file(latest_checkpoint / 'model.safetensors')
model.load_state_dict(state_dict)
model.to(device)
model.eval()

print('Loading dataset.')
dataset = load_dataset(config.dataset_id)['train']

print(
    f'Computing {config.p_virality_threshold} percentile combined threshold.')
engagement_scores = np.array(dataset['engagement_score'])
velocity_scores = np.array(dataset['view_velocity_score'])
combined_scores = engagement_scores / engagement_scores.max() + velocity_scores / \
    velocity_scores.max()
combined_threshold = np.quantile(combined_scores, config.p_virality_threshold)
print(f'Combined threshold: {combined_threshold:.4f}')


def is_viral(engagement, velocity):
    return (engagement / engagement_scores.max() + velocity / velocity_scores.max()) >= combined_threshold


example_idx = np.random.randint(0, len(dataset))
example = dataset[example_idx]

print(f'\n=== Running inference on example {example_idx} ===')
print(f'Ground truth engagement score: {example["engagement_score"]:.4f}')
print(f'Ground truth velocity score: {example["view_velocity_score"]:.4f}')
print(
    f'Ground truth is_viral: {is_viral(example["engagement_score"], example["view_velocity_score"])}')

processor = DataProcessor(config)

batch = {
    key: [example[key]] if not isinstance(
        example[key], bytes) else [example[key]]
    for key in example.keys()
}
processed = processor.process_batch(batch)
inference_batch = {
    k: v.to(device) if isinstance(v, torch.Tensor) else v
    for k, v in processed.items()
    if k != 'labels'
}

with torch.no_grad():
    output = model(**inference_batch, labels=None)
    logits = output['logits']

pred_engagement, pred_velocity = np.expm1(logits[0].cpu().numpy())
ground_truth_viral = is_viral(
    example["engagement_score"], example["view_velocity_score"])
pred_is_viral = is_viral(pred_engagement, pred_velocity)

print(f'\n=== Predictions ===')
print(f'Predicted engagement score: {pred_engagement:.4f}')
print(f'Predicted velocity score: {pred_velocity:.4f}')
print(f'Predicted is_viral: {pred_is_viral}')

print(f'\n=== Summary ===')
print(f'Ground truth: {"VIRAL" if ground_truth_viral else "NOT VIRAL"}')
print(f'Prediction:  {"VIRAL" if pred_is_viral else "NOT VIRAL"}')
print(f'Match: {ground_truth_viral == pred_is_viral}')

print('\n=== Video ===')
video_bytes = example['video_bytes']
if video_bytes:
    with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as temp_file:
        temp_file.write(video_bytes)
        temp_path = temp_file.name
    try:
        clip = VideoFileClip(temp_path)
        print(
            f'Video specs - Size: {clip.size}, Duration: {clip.duration:.2f}s, FPS: {clip.fps}')
        clip.close()
        display(Video(temp_path, embed=True))
    finally:
        os.unlink(temp_path)
else:
    print('No video bytes available')

Using MPS.
Loading model from: data/checkpoints/checkpoint-168.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/184 [00:00<?, ?it/s]

VideoMAEModel LOAD REPORT from: MCG-NJU/videomae-base
Key                                                                  | Status     |  | 
---------------------------------------------------------------------+------------+--+-
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.key.weight   | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_after.bias             | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_before.bias            | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.v_bias       | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.output.dense.bias                | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.value.weight | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_after.weight           | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_before.weight          | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.output.dense.weight 

Loading dataset.
Computing 0.95 percentile combined threshold.
Combined threshold: 1.4510

=== Running inference on example 389 ===
Ground truth engagement score: 0.1052
Ground truth velocity score: 23.8913
Ground truth is_viral: False


The image processor of type `VideoMAEImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
`use_fast` is set to `True` but the image processor class does not have a fast version.  Falling back to the slow version.



=== Predictions ===
Predicted engagement score: 1.1037
Predicted velocity score: 22.4898
Predicted is_viral: True

=== Summary ===
Ground truth: NOT VIRAL
Prediction:  VIRAL
Match: False

=== Video ===
Video specs - Size: [224, 224], Duration: 60.06s, FPS: 16.0
